In [4]:
import pandas as pd
import numpy as np
import sqlite3
import re
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('./data', exist_ok=True)
os.makedirs('./outputs', exist_ok=True)

In [6]:
df = pd.read_csv("../data/raw/fake_job_postings.csv")

# The dataset already has a job_id column — confirm it
print(df.columns.tolist())
print(f"Shape: {df.shape}")
print(f"Fraud rate: {df['fraudulent'].mean()*100:.1f}%")

# Fill all text columns with empty string
text_cols = ['company_profile','description','requirements',
             'benefits','title','location','department',
             'salary_range','employment_type',
             'required_experience','required_education',
             'industry','function']

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].fillna('').astype(str).str.strip()

# Create flag columns (1 = present, 0 = missing)
df['has_company_profile'] = (df['company_profile'] != '').astype(int)
df['has_salary']          = (df['salary_range']    != '').astype(int)
df['has_requirements']    = (df['requirements']    != '').astype(int)
df['has_benefits']        = (df['benefits']        != '').astype(int)
df['has_logo']            = df['has_company_logo'].fillna(0).astype(int)
df['telecommuting']       = df['telecommuting'].fillna(0).astype(int)

# Combine all text into one field for NLP
df['full_text'] = (df['title'] + ' ' +
                   df['company_profile'] + ' ' +
                   df['description'] + ' ' +
                   df['requirements'])

print(" Cleaning done")
print(df[['job_id','title','fraudulent']].head(3))

['job_id', 'title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'fraudulent']
Shape: (17880, 18)
Fraud rate: 4.8%
 Cleaning done
   job_id                                      title  fraudulent
0       1                           Marketing Intern           0
1       2  Customer Service - Cloud Video Production           0
2       3    Commissioning Machinery Assistant (CMA)           0


In [7]:
def parse_salary(s):
    nums = re.findall(r'\d+', str(s).replace(',',''))
    if len(nums) >= 2:
        return (int(nums[0]) + int(nums[1])) / 2
    return np.nan

df['salary_midpoint'] = df['salary_range'].apply(parse_salary)

conn = sqlite3.connect('./data/job_scam.db')

df[['job_id','title','location','department',
    'salary_range','employment_type','fraudulent']].to_sql(
    'job_postings', conn, if_exists='replace', index=False)

df[['job_id','company_profile','has_company_profile',
    'has_logo','telecommuting']].to_sql(
    'company_info', conn, if_exists='replace', index=False)

df[['job_id','description','requirements','benefits']].to_sql(
    'job_description', conn, if_exists='replace', index=False)

df[['job_id','required_experience',
    'required_education','industry','function']].to_sql(
    'job_requirements', conn, if_exists='replace', index=False)

df[['job_id','salary_range','salary_midpoint',
    'has_salary']].to_sql(
    'salary_analysis', conn, if_exists='replace', index=False)

df[['job_id','fraudulent']].to_sql(
    'fraud_labels', conn, if_exists='replace', index=False)

print(" All 6 tables written to ./data/job_scam.db")

# Run the 3 analytical queries
queries = {
    "fraud_by_industry": """
        SELECT jr.industry,
               COUNT(jp.job_id)                  AS total_postings,
               SUM(fl.fraudulent)                AS fraud_count,
               ROUND(AVG(fl.fraudulent)*100,1)   AS fraud_rate_pct
        FROM job_postings jp
        JOIN job_requirements jr ON jp.job_id = jr.job_id
        JOIN fraud_labels     fl ON jp.job_id = fl.job_id
        WHERE jr.industry != '' AND jr.industry != 'nan'
        GROUP BY jr.industry
        HAVING total_postings > 50
        ORDER BY fraud_rate_pct DESC
        LIMIT 15
    """,
    "missing_fields_vs_fraud": """
        SELECT ci.has_company_profile,
               ci.has_logo,
               sa.has_salary,
               COUNT(*)                          AS total,
               ROUND(AVG(fl.fraudulent)*100,1)   AS fraud_rate_pct
        FROM company_info ci
        JOIN salary_analysis sa ON ci.job_id = sa.job_id
        JOIN fraud_labels    fl ON ci.job_id = fl.job_id
        GROUP BY ci.has_company_profile, ci.has_logo, sa.has_salary
        ORDER BY fraud_rate_pct DESC
    """,
    "fraud_by_experience": """
        SELECT jr.required_experience,
               COUNT(*)                          AS postings,
               ROUND(AVG(fl.fraudulent)*100,1)   AS fraud_rate_pct
        FROM job_requirements jr
        JOIN fraud_labels fl ON jr.job_id = fl.job_id
        WHERE jr.required_experience != ''
          AND jr.required_experience != 'nan'
        GROUP BY jr.required_experience
        ORDER BY fraud_rate_pct DESC
    """
}

for name, q in queries.items():
    result = pd.read_sql(q, conn)
    result.to_csv(f'./data/{name}.csv', index=False)
    print(f"\n── {name} ──")
    print(result.to_string(index=False))

conn.close()
print("\n SQL queries done — CSVs saved in ./data/")

 All 6 tables written to ./data/job_scam.db

── fraud_by_industry ──
                           industry  total_postings  fraud_count  fraud_rate_pct
                       Oil & Energy             287          109            38.0
                         Accounting             159           57            35.8
          Leisure, Travel & Tourism              76           21            27.6
                        Hospitality              88           14            15.9
                        Real Estate             175           24            13.7
       Health, Wellness and Fitness             127           15            11.8
             Hospital & Health Care             497           51            10.3
                 Telecommunications             342           26             7.6
                      Entertainment              74            5             6.8
                  Consumer Services             358           24             6.7
            Staffing and Recruiting     